In [1]:
%matplotlib inline

import os
import sys
import math

from sys import platform

sys.path.append('../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken

import torch
import torch.nn as nn



%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from gpt_download import download_and_load_gpt2
from llm_from_scratch.CH5.utils import load_weights_into_gpt

In [2]:
base_tokenizer=tiktoken.get_encoding("gpt2")
sample_text="Hello, MyNewToken_1 is a new token. <|endoftext|>"
token_ids=base_tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})

In [3]:
for token_id in token_ids: print(f"{token_id} - > {base_tokenizer.decode([token_id])}")

15496 - > Hello
11 - > ,
2011 - >  My
3791 - > New
30642 - > Token
62 - > _
16 - > 1
318 - >  is
257 - >  a
649 - >  new
11241 - >  token
13 - > .
220 - >  
50256 - > <|endoftext|>


In [4]:
# define custom tokens and their token ids
custom_tokens=["MyNewToken_1", "MyNewToken_2"]
custom_token_ids={token:base_tokenizer.n_vocab+i for i, token in enumerate(custom_tokens)}
print(f"{custom_token_ids=}")

custom_token_ids={'MyNewToken_1': 50257, 'MyNewToken_2': 50258}


In [5]:
print(f"{base_tokenizer._pat_str=}")
print(f"{list(base_tokenizer._mergeable_ranks.keys())[:5]=},\n{list(base_tokenizer._mergeable_ranks.values())[:5]=}")

base_tokenizer._pat_str="'(?:[sdmt]|ll|ve|re)| ?\\p{L}++| ?\\p{N}++| ?[^\\s\\p{L}\\p{N}]++|\\s++$|\\s+(?!\\S)|\\s"
list(base_tokenizer._mergeable_ranks.keys())[:5]=[b'!', b'"', b'#', b'$', b'%'],
list(base_tokenizer._mergeable_ranks.values())[:5]=[0, 1, 2, 3, 4]


In [6]:
# create a new encoding object with extended tokens
extended_tokenizer=tiktoken.Encoding(
    name="gpt2_custom", pat_str=base_tokenizer._pat_str, mergeable_ranks=base_tokenizer._mergeable_ranks,
    special_tokens={**base_tokenizer._special_tokens, **custom_token_ids}
)

In [7]:
special_tokens_set=set(custom_tokens)|{"<|endoftext|>"}
token_ids=extended_tokenizer.encode(
    "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>",
    allowed_special=special_tokens_set
)
print(token_ids)

[36674, 2420, 351, 220, 50257, 290, 220, 50258, 13, 220, 50256]


In [8]:
for token_id in token_ids: print(f"{token_id} -> {extended_tokenizer.decode([token_id])}")

36674 -> Sample
2420 ->  text
351 ->  with
220 ->  
50257 -> MyNewToken_1
290 ->  and
220 ->  
50258 -> MyNewToken_2
13 -> .
220 ->  
50256 -> <|endoftext|>


In [9]:
main_output_dirpath='../../../../../../results'
settings, params=download_and_load_gpt2(model_size='124M', models_dir=f"{main_output_dirpath}/gpts")

File already exists and is up-to-date: ../../../../../../results/gpts/124M/checkpoint
File already exists and is up-to-date: ../../../../../../results/gpts/124M/encoder.json
File already exists and is up-to-date: ../../../../../../results/gpts/124M/hparams.json
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.index
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.meta
File already exists and is up-to-date: ../../../../../../results/gpts/124M/vocab.bpe


In [10]:
GPT_CONFIG_124M={
    "vocab_size":50257,       # vocabulary size
    "context_length":256,     # shortened context length (originally 1024)
    "emb_dim":768,            # embedding dimension
    "n_heads":12,             # number of attention heads
    "n_layers":12,            # number of layers
    "drop_rate":0.1,          # dropout rate
    "qkv_bias":False          # query-key-value bias
}
# define model configurations in a dictionary for compactness
model_configs={
    "gpt2-small (124M)":{"emb_dim":768, "n_layers":12, "n_heads":12},
    "gpt2-medium (355M)":{"emb_dim":1024, "n_layers":24, "n_heads":16},
    "gpt2-large (774M)":{"emb_dim":1280, "n_layers":36, "n_heads":20},    
    "gpt2-xl (1558M)":{"emb_dim":1600, "n_layers":48, "n_heads":25},
}
# copy the base configuration and update with specific model settings
model_name="gpt2-small (124M)" # example model name
NEW_CONFIG=GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length":1024, "qkv_bias":True})

gpt=GPTModel(NEW_CONFIG)
load_weights_into_gpt(gpt, params)
gpt.eval();

### Using the pretrained GPT model

In [11]:
sample_text="Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>"
original_token_ids=base_tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})
new_token_ids=extended_tokenizer.encode(sample_text, allowed_special=special_tokens_set)
print(f"{len(original_token_ids)=}, {original_token_ids}")
print(f"{len(new_token_ids)=}, {new_token_ids}")

len(original_token_ids)=17, [36674, 2420, 351, 2011, 3791, 30642, 62, 16, 290, 2011, 3791, 30642, 62, 17, 13, 220, 50256]
len(new_token_ids)=11, [36674, 2420, 351, 220, 50257, 290, 220, 50258, 13, 220, 50256]


In [12]:
with torch.no_grad(): out=gpt(torch.tensor([original_token_ids]))

print(f"{out.shape=}")# {out}")

out.shape=torch.Size([1, 17, 50257])


In [13]:
with torch.no_grad(): gpt(torch.tensor([new_token_ids]))

IndexError: index out of range in self

GPT expects a fixed vocabulary size via its input embedding layer and its output layer

### Update the embedding layer

In [14]:
gpt.tok_emb

Embedding(50257, 768)

In [15]:
num_tokens, emb_size=gpt.tok_emb.weight.shape
new_num_tokens=num_tokens+2

# create a new embedding layer
new_embedding=torch.nn.Embedding(new_num_tokens, emb_size)
print(f"{new_embedding.weight.shape=}")

# copy weights from the old embedding layer
new_embedding.weight.data[:num_tokens]=gpt.tok_emb.weight.data

# replace the old embedding layer with the new one in the model
gpt.tok_emb=new_embedding

print(f"{gpt.tok_emb.weight.shape=}")

new_embedding.weight.shape=torch.Size([50259, 768])
gpt.tok_emb.weight.shape=torch.Size([50259, 768])


### Updating the output layer

In [17]:
print(f"{gpt.out_head=}\n{gpt.out_head.weight.shape=}")

gpt.out_head=Linear(in_features=768, out_features=50257, bias=False)
gpt.out_head.weight.shape=torch.Size([50257, 768])


In [18]:
original_out_features, original_in_features=gpt.out_head.weight.shape

# define the new number of output features (e.g., adding 2 new tensors)
new_out_features=original_out_features+2

# create a new linear layer with the extended output size
new_linear=torch.nn.Linear(original_in_features, new_out_features)

# copy the weights and biases fromn the original linear layer
with torch.no_grad():
    new_linear.weight[:original_out_features]=gpt.out_head.weight
    if gpt.out_head.bias is not None: new_linear.bias[:original_out_features]=gpt.out_head.bias

# replace the original layer with the new one
gpt.out_head=new_linear

print(gpt.out_head)

Linear(in_features=768, out_features=50259, bias=True)


In [19]:
with torch.no_grad(): output=gpt(torch.tensor([original_token_ids]))
print(output)

tensor([[[-29.9766, -29.2601, -31.9121,  ..., -29.4867,  -2.1226,   1.8911],
         [-95.8014, -94.9176, -98.8957,  ..., -94.5866,  -8.3026,   7.3584],
         [-90.4549, -89.3511, -91.4690,  ..., -89.5897,  -8.2997,   6.3413],
         ...,
         [-89.7071, -90.3555, -88.7285,  ..., -83.2528,  -8.3021,   6.6913],
         [-48.2988, -49.0238, -50.4875,  ..., -47.2997,  -4.6278,   4.7211],
         [-86.7040, -79.5252, -82.1754,  ..., -87.8273,  -5.8662,   7.4121]]])


In [20]:
with torch.no_grad(): output=gpt(torch.tensor([new_token_ids]))
print(output)

tensor([[[ -29.9766,  -29.2601,  -31.9121,  ...,  -29.4867,   -2.1226,
             1.8911],
         [ -95.8014,  -94.9176,  -98.8957,  ...,  -94.5866,   -8.3026,
             7.3584],
         [ -90.4549,  -89.3511,  -91.4690,  ...,  -89.5897,   -8.2997,
             6.3413],
         ...,
         [-107.0999, -106.0766, -106.1929,  ..., -100.0982,   -8.6381,
             7.8349],
         [ -75.3693,  -73.8104,  -76.2728,  ...,  -73.4015,   -5.7641,
             6.3366],
         [-100.4480,  -93.2063,  -96.7943,  ..., -100.7501,   -6.7183,
             8.0363]]])


In practice, we want to finetune (or continually pretrain) the model (specifically the new embedding and output layer)
on data containing the new tokens

#### In case of weight tying

In [ ]:
gpt.out_head.weight=gpt.tok_emb.weight